## Data In Hand

In [1]:
pip install tcgdex-sdk

Note: you may need to restart the kernel to use updated packages.


In [86]:
import pandas as pd
import requests

#Fetching all English cards
baseUrl = "https://api.tcgdex.net/v2/en/cards/"

df = pd.read_json(baseUrl)
df.head()

,id,localId,name,image
0,exu-!,!,Unown,NaN
1,exu-%3F,%3F,Unown,NaN
2,bw1-1,1,Snivy,https://assets.tcgdex.net/en/bw/bw1/1
3,dv1-1,1,Dratini,https://assets.tcgdex.net/en/bw/dv1/1
4,bw2-1,1,Pansage,https://assets.tcgdex.net/en/bw/bw2/1


The info about TCGDex's API is here: https://tcgdex.dev/rest

The code above only grabs the card data from https://api.tcgdex.net/v2/en/cards/. If I want to have the data about card types, I will first have to collect every pack ID and grab all data from those.

Code for reordering columns comes from here: https://www.geeksforgeeks.org/python/change-the-order-of-a-pandas-dataframe-columns-in-python/

In [88]:
#Fetching all English sets
url = "https://api.tcgdex.net/v2/en/sets/"

setDf = pd.read_json(url)

#normalizes cardCount column
normalized = pd.json_normalize(setDf['cardCount'])

#adds to dataframe
setDf = pd.concat([setDf.drop(columns=['cardCount']), normalized], axis=1)
#renaming columns
setDf = setDf.rename(columns={'total': 'totalCards', 'official': 'officialCards'})

#reorder columns
setDf = setDf.iloc[:, [0, 1, 4, 5, 2, 3]] 

setDf.head()

,id,name,totalCards,officialCards,logo,symbol
0,miscp,Miscellaneous Promos,1,1,NaN,NaN
1,base1,Base Set,102,102,https://assets.tcgdex.net/en/base/base1/logo,NaN
2,base2,Jungle,64,64,https://assets.tcgdex.net/en/base/base2/logo,https://assets.tcgdex.net/univ/base/base2/symbol
3,basep,Wizards Black Star Promos,53,53,https://assets.tcgdex.net/en/base/basep/logo,https://assets.tcgdex.net/univ/base/basep/symbol
4,wp,W Promotional,7,7,NaN,NaN


In [89]:
#makes chunkDf (url bit) that is copied from df[id]
#this will make us able to scrape the data from each card
urlDf = df[['id']].copy()

#replaces all '-' in df with '/'
#used https://www.geeksforgeeks.org/python/replace-characters-in-strings-in-pandas-dataframe/ to get code
urlDf = urlDf.replace('-', '/', regex=True)
urlDf.head()

,id
0,exu/!
1,exu/%3F
2,bw1/1
3,dv1/1
4,bw2/1


The code below gets the data from each url for each set.

In [94]:
rows = []

#NOTE: for the Data In Hand submission, this code is going to only scrape the first 20 rows of urlDf
#This is going to be a massive dataset so I don't want to require viewers to sit through the full scraping process
#I plan on creating a spreadsheet/csv file with all of the gathered data for the actual project analysis
for cardSet in urlDf["id"][:21]:
    cardUrl = f"{url}{cardSet}"
    response = requests.get(cardUrl)
    if response.status_code == 200:
        data = response.json()
        rows.append(data)
        print(f"{cardSet} scraped!")

exu/! scraped!
bw1/1 scraped!
dv1/1 scraped!
bw2/1 scraped!
bw3/1 scraped!
bw4/1 scraped!
bw5/1 scraped!
bw10/1 scraped!
bw8/1 scraped!
bw7/1 scraped!
bw9/1 scraped!
bw11/1 scraped!
bw6/1 scraped!
si1/1 scraped!
neo4/1 scraped!
neo2/1 scraped!
neo3/1 scraped!
neo1/1 scraped!
sv10/001 scraped!
sv01/001 scraped!


In [95]:
cardsDf = pd.json_normalize(rows)
cardsDf

,category,id,illustrator,localId,name,rarity,variants_detailed,dexId,hp,types,...,pricing.tcgplayer.1st-edition-holofoil.highPrice,pricing.tcgplayer.1st-edition-holofoil.marketPrice,pricing.tcgplayer.1st-edition-holofoil.directLowPrice,pricing.tcgplayer.unlimited-holofoil.productId,pricing.tcgplayer.unlimited-holofoil.lowPrice,pricing.tcgplayer.unlimited-holofoil.midPrice,pricing.tcgplayer.unlimited-holofoil.highPrice,pricing.tcgplayer.unlimited-holofoil.marketPrice,pricing.tcgplayer.unlimited-holofoil.directLowPrice,regulationMark
0,Pokemon,exu-!,Kyoko Koizumi,!,Unown,Rare,"[{'type': 'holo', 'size': 'standard', 'thirdPa...",[201],60,[Psychic],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Pokemon,bw1-1,Kagemaru Himeno,1,Snivy,Common,"[{'type': 'normal', 'size': 'standard', 'third...",[495],60,[Grass],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Pokemon,dv1-1,Masakazu Fukuda,1,Dratini,Rare,"[{'type': 'holo', 'size': 'standard', 'thirdPa...",[147],40,[Dragon],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Pokemon,bw2-1,Kouki Saitou,1,Pansage,Common,"[{'type': 'normal', 'size': 'standard', 'third...",[511],70,[Grass],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Pokemon,bw3-1,Mizue,1,Sewaddle,Common,"[{'type': 'normal', 'size': 'standard', 'third...",[540],50,[Grass],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Pokemon,bw4-1,Masakazu Fukuda,1,Pinsir,Rare,"[{'type': 'normal', 'size': 'standard', 'third...",[127],80,[Grass],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Pokemon,bw5-1,Mitsuhiro Arita,1,Bulbasaur,Common,"[{'type': 'normal', 'size': 'standard', 'third...",[1],60,[Grass],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Pokemon,bw10-1,Naoyo Kimura,1,Surskit,Common,"[{'type': 'normal', 'size': 'standard', 'varia...",[283],50,[Grass],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Pokemon,bw8-1,Kouki Saitou,1,Turtwig,Common,"[{'type': 'normal', 'size': 'standard', 'varia...",[387],70,[Grass],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Pokemon,bw7-1,Kanako Eo,1,Oddish,Common,"[{'type': 'normal', 'size': 'standard', 'third...",[43],50,[Grass],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Data structure
There are numbers for me to analyze with. I could use the selling price or edit the dataset further and all that. I will also edit the database to do stuff like count variants

In [96]:
cardsDf.columns

Index(['category', 'id', 'illustrator', 'localId', 'name', 'rarity',
       'variants_detailed', 'dexId', 'hp', 'types', 'stage', 'abilities',
       'attacks', 'weaknesses', 'retreat', 'updated', 'set.cardCount.official',
       'set.cardCount.total', 'set.id', 'set.name', 'variants.firstEdition',
       'variants.holo', 'variants.normal', 'variants.reverse',
       'variants.wPromo', 'legal.standard', 'legal.expanded',
       'pricing.cardmarket.updated', 'pricing.cardmarket.unit',
       'pricing.cardmarket.idProduct', 'pricing.cardmarket.avg',
       'pricing.cardmarket.low', 'pricing.cardmarket.trend',
       'pricing.cardmarket.avg1', 'pricing.cardmarket.avg7',
       'pricing.cardmarket.avg30', 'pricing.cardmarket.avg-holo',
       'pricing.cardmarket.low-holo', 'pricing.cardmarket.trend-holo',
       'pricing.cardmarket.avg1-holo', 'pricing.cardmarket.avg7-holo',
       'pricing.cardmarket.avg30-holo', 'pricing.tcgplayer.unit',
       'pricing.tcgplayer.updated', 'pricing.tcgpl